# 🔮 Notebook 2 — Prophet Forecasting

Facebook Prophet is an additive decomposition model designed for business time series.
It handles trend, seasonality, and holidays out of the box with minimal tuning.

## 2.1 Setup

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.join('..', 'src'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from data_preprocessing import load_and_clean, aggregate_monthly, train_test_split_ts
from prophet_model import (train_prophet, forecast_prophet,
                            evaluate_on_test, plot_prophet_forecast, plot_prophet_components)
from evaluate import evaluate_model

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

os.makedirs('../outputs', exist_ok=True)


## 2.2 Load & Split Data

In [ ]:
df = load_and_clean('../data/train.csv')
monthly = aggregate_monthly(df)

TEST_MONTHS = 6
train_df, test_df = train_test_split_ts(monthly, test_months=TEST_MONTHS)
print(f"Train: {len(train_df)} months | Test: {len(test_df)} months")
print("Train range:", train_df['ds'].min().date(), "→", train_df['ds'].max().date())
print("Test range: ", test_df['ds'].min().date(),  "→", test_df['ds'].max().date())


## 2.3 Train Prophet

Key hyperparameters:
- `changepoint_prior_scale` — controls trend flexibility (default 0.05, we use 0.1)
- `seasonality_mode='multiplicative'` — better for growing series
- Custom monthly seasonality added via `add_seasonality()`

In [ ]:
model = train_prophet(
    train_df,
    yearly_seasonality=True,
    weekly_seasonality=False,
    monthly_seasonality=True,
    changepoint_prior_scale=0.1
)


## 2.4 Generate Forecast

In [ ]:
forecast = forecast_prophet(model, periods=TEST_MONTHS, freq='MS')
# Preview the last 10 rows — these include the future forecast
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10)


## 2.5 Visualize Forecast

In [ ]:
plot_prophet_forecast(
    model, forecast, train_df, test_df,
    save_path='../outputs/prophet_forecast.png'
)


## 2.6 Decomposed Components

Prophet's biggest interpretability advantage — you can see *why* it makes each prediction.

In [ ]:
plot_prophet_components(
    model, forecast,
    save_path='../outputs/prophet_components.png'
)


## 2.7 Evaluate on Test Set

In [ ]:
prophet_preds = evaluate_on_test(forecast, test_df)
result = evaluate_model('Prophet', test_df['y'].values, prophet_preds)

# Show actuals vs predictions table
comparison = test_df[['ds', 'y']].copy()
comparison['Prophet_Predicted'] = prophet_preds
comparison['Error'] = comparison['y'] - comparison['Prophet_Predicted']
comparison['Error_%'] = (comparison['Error'] / comparison['y'] * 100).round(2)
comparison.rename(columns={'y': 'Actual'}, inplace=True)
comparison


## 2.8 Hyperparameter Sensitivity

Test different `changepoint_prior_scale` values to see how they affect accuracy.

In [ ]:
from prophet import Prophet

results = []
for cp in [0.01, 0.05, 0.1, 0.3, 0.5]:
    m = train_prophet(train_df, changepoint_prior_scale=cp)
    fc = forecast_prophet(m, periods=TEST_MONTHS)
    preds = evaluate_on_test(fc, test_df)
    mae = np.mean(np.abs(test_df['y'].values - preds))
    results.append({'changepoint_prior_scale': cp, 'MAE': round(mae, 2)})

import pandas as pd
sensitivity_df = pd.DataFrame(results)
print(sensitivity_df.to_string(index=False))
print(f"\nBest CPS: {sensitivity_df.loc[sensitivity_df['MAE'].idxmin(), 'changepoint_prior_scale']}")


## 2.9 Prophet — Pros & Cons

| ✅ Pros | ❌ Cons |
|---|---|
| Extremely interpretable (components plot) | Assumes additive/multiplicative structure |
| Handles missing data natively | Cannot learn complex non-linear patterns |
| Built-in seasonality + holiday support | May underfit highly volatile series |
| No feature engineering required | Less flexible than deep learning |
| Fast training (< 1 second) | |